> 旧Ultralytics版の記録です。現在のYOLOv5実験A〜Gはリポジトリの basketball_tracking_midterm/README.md を参照してください。データ・重みは別途必要です。experiment10は現在、開発評価用です。

# ball検出：YOLO26nの追加学習（Google Colab）

このノートブックと `ball_yolo_colab.zip` を使います。先に「ランタイム → ランタイムのタイプを変更 → GPU」を選択してください。

元の分割を保持：train 300枚／val 50枚／test 58枚。ボールあり247枚、確認済み背景161枚。クラスは `0: ball` のみです。

`colab_ball_yolo` フォルダをマイドライブ直下にコピーし、上から実行します。ZIPだけを `MyDrive/ball_yolo/` またはマイドライブ直下に置く方法にも対応します。自動実行されるのは追加学習1回です。最終testは初期状態では実行しません。

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
import zipfile, shutil, subprocess, sys, json
# フォルダごとのコピーとZIPだけのアップロードに対応。
# 別の場所を使う場合は ZIP_PATH に実際のフルパスを指定してください。
ZIP_CANDIDATES = [
    Path("/content/drive/MyDrive/colab_ball_yolo/ball_yolo_colab.zip"),
    Path("/content/drive/MyDrive/ball_yolo/ball_yolo_colab.zip"),
    Path("/content/drive/MyDrive/ball_yolo_colab.zip"),
]
ZIP_PATH = next((p for p in ZIP_CANDIDATES if p.is_file()), None)
if ZIP_PATH is None:
    raise FileNotFoundError(
        "ZIPが見つかりません。Driveの配置場所を確認してください。検索した場所:\n"
        + "\n".join(str(p) for p in ZIP_CANDIDATES)
    )
print("使用するZIP:", ZIP_PATH)
WORK = Path("/content/ball_yolo_colab")
RUNS = Path("/content/drive/MyDrive/ball_yolo/runs")
assert ZIP_PATH.is_file(), f"ZIPを配置してください: {ZIP_PATH}"
# 初回のみ展開。再接続時も同じ /content パスを使用します。
if not WORK.exists():
    with zipfile.ZipFile(ZIP_PATH) as archive:
        target = Path("/content").resolve()
        for member in archive.infolist():
            dest = (target / member.filename).resolve()
            if not dest.is_relative_to(target) or not member.filename.startswith("ball_yolo_colab/"):
                raise ValueError("Unexpected ZIP path")
        archive.extractall(target)
assert (WORK / "dataset/manifest.csv").is_file()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(WORK / "requirements-colab.txt")], check=True)
import torch, ultralytics
print("torch", torch.__version__, "ultralytics", ultralytics.__version__)
assert torch.cuda.is_available(), "GPUランタイムを選択してください"
print(torch.cuda.get_device_name(0))

## データの整合性・正解ラベルの見た目を確認

元画像・ラベルのSHA-256、対応関係、座標を検証します。緑の枠がballです。背景画像には枠がありません。

In [ ]:
subprocess.run([sys.executable, str(WORK / "prepare_dataset.py"), "--verify", str(WORK / "dataset")], check=True)
report = json.loads((WORK / "dataset/validation_report.json").read_text())
print(json.dumps(report["splits"], indent=2))
from IPython.display import display, Image
for split in ("train", "val"):
    display(Image(filename=str(WORK / f"preview_{split}.jpg")))

## 追加学習

既存プロジェクトと同じCOCO事前学習済み `yolo26n.pt` を自動取得して、ball 1クラスに追加学習します。初期設定は100 epochs、早期終了20 epochs、入力960、batch 8です。小さいボールのため解像度をやや高めにしていますが、最適値を保証する設定ではありません。

GPUメモリ不足なら `BATCH=4` または `2` に下げます。結果は各epochごとにDriveへ保存します。

In [ ]:
from datetime import datetime, timezone
MODEL = "yolo26n.pt"
IMGSZ = 960
BATCH = 8
EPOCHS = 100
LR0 = 0.001
RUN_NAME = "ball_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_DIR = RUNS / RUN_NAME
cmd = [sys.executable, str(WORK / "train_yolo.py"), "train",
       "--dataset", str(WORK / "dataset"), "--project", str(RUNS),
       "--name", RUN_NAME, "--model", MODEL, "--epochs", str(EPOCHS),
       "--imgsz", str(IMGSZ), "--batch", str(BATCH), "--lr0", str(LR0)]
print("保存先:", RUN_DIR)
subprocess.run(cmd, check=True)
BEST = RUN_DIR / "weights/best.pt"
assert BEST.is_file()

## 検証用データで結果を確認

mAP50-95を中心にPrecision・Recallと見逃し画像を比較します。変更する場合は上の学習セルの `IMGSZ`（640/960/1280）や `LR0`（0.0005/0.001）を一つずつ変更して新しい学習を実行します。50枚と少ないため、僅かな差だけで優劣を決めないでください。testで設定を選ばないでください。

In [ ]:
import pandas as pd
history = pd.read_csv(RUN_DIR / "results.csv")
history.columns = history.columns.str.strip()
display(history.tail())
for name in ("results.png", "val_batch0_labels.jpg", "val_batch0_pred.jpg"):
    p = RUN_DIR / name
    if p.exists(): display(Image(filename=str(p)))
subprocess.run([sys.executable, str(WORK / "train_yolo.py"), "val",
                "--dataset", str(WORK / "dataset"), "--project", str(RUNS),
                "--model", str(BEST), "--imgsz", str(IMGSZ), "--batch", str(BATCH)], check=True)

## 任意：切断後の再開

新しいランタイムでは最初の2つのコードセルを実行してデータを再展開し、下のパスを既存runの `last.pt` に変更して `RESUME=True` にします。追加学習セルは再実行しません。完了済みrunへのresumeは使えません。

In [ ]:
RESUME = False
LAST = Path("/content/drive/MyDrive/ball_yolo/runs/ここを実際のrun名に変更/weights/last.pt")
if RESUME:
    assert LAST.is_file()
    subprocess.run([sys.executable, str(WORK / "train_yolo.py"), "train",
                    "--dataset", str(WORK / "dataset"), "--resume", str(LAST)], check=True)
    RUN_DIR = LAST.parent.parent
    BEST = RUN_DIR / "weights/best.pt"
    import yaml
    saved_args = yaml.safe_load((RUN_DIR / "args.yaml").read_text())
    IMGSZ = saved_args["imgsz"]
    BATCH = saved_args["batch"]

## 設定を確定した後だけ最終test

採用する重みを `SELECTED_BEST` に指定してください。比較したモデルの中からvalの結果で選び、入力サイズもそのモデルと同じにします。`RUN_FINAL_TEST=True` に変更すると58枚で最終評価を実行します。この結果を使って再調整すると独立した最終評価ではなくなります。

In [ ]:
RUN_FINAL_TEST = False
SELECTED_BEST = BEST
if RUN_FINAL_TEST:
    subprocess.run([sys.executable, str(WORK / "train_yolo.py"), "test", "--final-test",
                    "--dataset", str(WORK / "dataset"), "--project", str(RUNS),
                    "--model", str(SELECTED_BEST), "--imgsz", str(IMGSZ), "--batch", str(BATCH)], check=True)

## モデルを取り出す

`best.pt` と学習設定・指標・環境情報はDriveのrunフォルダに残ります。以下は選択した重みのダウンロード用です。既存プログラムでは `--model weights/ball_best.pt --class-id 0` を指定します（COCOのsports ball番号32とは異なります）。

In [ ]:
DOWNLOAD = False
print("学習済みモデル:", SELECTED_BEST)
if DOWNLOAD:
    from google.colab import files
    files.download(str(SELECTED_BEST))

参考： [Ultralytics学習](https://docs.ultralytics.com/modes/train/) ／ [検出データ形式](https://docs.ultralytics.com/datasets/detect/)

このノートブックの学習セルは準備時点では未実行です。同一動画内の時間分割なので、別試合・別撮影環境への汎化性能は別データで評価してください。